In [ ]:
!pip install -q flask flask-cors pyngrok transformers \
    accelerate bitsandbytes sentencepiece torch

print("All dependencies installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.3 MB/s eta 0:00:00
All dependencies installed!


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

MODEL = "GRMenon/mental-health-mistral-7b-instructv0.2-finetuned-V2"
print(f"Downloading {MODEL}...")
print("This takes 8-12 minutes. No token needed, no access gate.")

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=quant,
    device_map="auto",
)
print("Model loaded. Brydge is awake.")

This takes 8-12 minutes. No token needed, no access gate.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/27.3M [00:00<?, ?B/s]

AttributeError: `weight` is not an nn.Module

In [ ]:
CRISIS_WORDS = [
    "suicide", "kill myself", "end my life", "want to die",
    "self harm", "hurt myself", "no point living", "disappear forever",
    "cant go on", "can't go on", "give up on life", "not worth living",
    "better off dead", "take my own life", "overdose"
]
CRISIS_RESPONSE = (
    "I hear you, and I'm really glad you told me. "
    "Please reach out to iCall right now — 📞 9152987821. "
    "You don't have to face this alone."
)

def is_crisis(text):
    return any(w in text.lower() for w in CRISIS_WORDS)

SYSTEM_PROMPT = open("listener.py").read()  # loaded from file below
print("Crisis detection and system prompt ready.")

Crisis detection and system prompt ready.


In [ ]:
from google.colab import files

print("Please upload 'brydge_prompt.txt'")
uploaded = files.upload()

if 'brydge_prompt.txt' in uploaded:
    print("'brydge_prompt.txt' uploaded successfully!")
else:
    print("File 'brydge_prompt.txt' not found in uploaded files. Please try again.")

Please upload 'brydge_prompt.txt'


Saving listener.py to listener.py
File 'brydge_prompt.txt' not found in uploaded files. Please try again.


In [ ]:
def chat(history, user_message, sphere="general"):
    history.append({"role": "user", "content": user_message})

    sphere_ctx = {
        "academic": "[Academic sphere — focus on exams, study, burnout]",
        "peer":     "[Peer sphere — focus on friendships, bullying, loneliness]",
        "family":   "[Family sphere — focus on conflict, expectations, home stress]",
    }.get(sphere, "")

    prompt = f"[INST] {SYSTEM_PROMPT}\n{sphere_ctx}\n\n"
    for m in history:
        r = "User" if m["role"] == "user" else "Brydge"
        prompt += f"{r}: {m['content']}\n"
    prompt += "[/INST]\nBrydge:"

    inputs = tokenizer(prompt, return_tensors="pt",
        truncation=True, max_length=3072).to("cuda")

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=180,
            temperature=0.72,
            do_sample=True,
            top_p=0.92,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id,
        )

    reply = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    for stop in ["User:", "Brydge:", "[INST]", "[/INST]"]:
        if stop in reply:
            reply = reply.split(stop)[0].strip()

    history.append({"role": "assistant", "content": reply})
    return reply, history

print("Chat function ready.")

Chat function ready.


In [ ]:
from flask import Flask, render_template_string, request, jsonify
import uuid

app = Flask(__name__)
sessions = {}

HTML = open("chat.html").read()  # the chat.html we built

@app.route("/")
def index():
    return render_template_string(HTML)

@app.route("/api/chat", methods=["POST"])
def chat_endpoint():
    data = request.json
    msg = data.get("message", "").strip()
    uid = data.get("user_id", str(uuid.uuid4()))
    sphere = data.get("sphere", "general")

    if not msg:
        return jsonify({"error": "No message"}), 400

    if uid not in sessions:
        sessions[uid] = []

    if is_crisis(msg):
        return jsonify({"response": CRISIS_RESPONSE, "crisis": True, "user_id": uid})

    response, sessions[uid] = chat(sessions[uid], msg, sphere)
    return jsonify({"response": response, "crisis": False, "user_id": uid})

@app.route("/api/reset", methods=["POST"])
def reset():
    data = request.json
    uid = data.get("user_id")
    if uid in sessions:
        sessions[uid] = []
    return jsonify({"status": "reset"})

print("Flask app defined.")

Flask app defined.


In [ ]:
from google.colab import files

print("Please upload 'chat.html'")
uploaded = files.upload()

if 'chat.html' in uploaded:
    print("'chat.html' uploaded successfully!")
else:
    print("File 'chat.html' not found in uploaded files. Please try again.")

Please upload 'chat.html'


Saving chat.html to chat.html
'chat.html' uploaded successfully!


In [1]:
import threading
import time
import subprocess
import re # For regex to extract URL

# Install localtunnel globally. This requires Node.js and npm,
# which are usually pre-installed in Google Colab environments.
print("Installing localtunnel...")
install_process = subprocess.run(
    ["npm", "install", "-g", "localtunnel"],
    capture_output=True, text=True, check=False
)
if install_process.returncode != 0:
    # npm install might fail if it's not in the PATH or permissions issues.
    # For simplicity, we just print the error and hope it works for most users.
    print(f"Warning: localtunnel npm install output:\n{install_process.stdout}\n{install_process.stderr}")
    print("If localtunnel fails below, ensure Node.js and npm are properly installed.")

# Start Flask app in a background thread
def run_flask_app():
    try:
        # 'app' is globally defined in a previous cell
        app.run(port=5000, use_reloader=False, debug=False)
    except Exception as e:
        print(f"Flask app failed to start: {e}")
        print("If you see 'Address already in use', please restart the Colab runtime (Runtime -> Restart runtime...) and try again.")

flask_thread = threading.Thread(target=run_flask_app)
flask_thread.daemon = True # Allows the main program to exit even if the thread is still running
flask_thread.start()

# Give Flask a moment to start up
time.sleep(3)

print("Attempting to start localtunnel to expose port 5000...")

public_url = None
lt_process = None
try:
    # Start localtunnel process
    # 'lt --port 5000' prints the URL to stdout
    lt_process = subprocess.Popen(
        ['lt', '--port', '5000'],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
    )

    # Wait for localtunnel to print the URL to stdout
    # It usually prints 'your url is: https://<random-subdomain>.loca.lt'
    timeout_start = time.time()
    timeout_seconds = 60 # Give it up to 60 seconds to connect
    while public_url is None and (time.time() - timeout_start) < timeout_seconds:
        line = lt_process.stdout.readline()
        if "your url is:" in line:
            public_url = line.split("your url is:")[1].strip()
            break
        elif "error:" in line.lower() or "Cannot connect to" in line:
            stderr_output = lt_process.stderr.read() # Read any accumulated stderr
            raise Exception(f"Localtunnel failed to start: {line.strip()} {stderr_output.strip()}")
        time.sleep(0.5) # Prevent busy-waiting

    if public_url:
        print("=" * 50)
        print("BRYDGE IS LIVE (via localtunnel)!")
        print(f"Open this URL: {public_url}")
        print("Share it with anyone — works on phone too!")
        print("=" * 50)
    else:
        # If no URL was found within the timeout
        raise Exception(f"Localtunnel failed to provide a public URL within {timeout_seconds} seconds. Stderr: {lt_process.stderr.read().strip()}")

except Exception as e:
    print(f"Error setting up localtunnel: {e}")
    print("Please ensure Node.js and npm are installed, and port 5000 is available (restart Colab runtime if needed).")
    # Make sure to terminate the localtunnel process if it was started and still running
    if lt_process and lt_process.poll() is None:
        lt_process.terminate()



Installing localtunnel...
Flask app failed to start: name 'app' is not defined
If you see 'Address already in use', please restart the Colab runtime (Runtime -> Restart runtime...) and try again.
Attempting to start localtunnel to expose port 5000...
BRYDGE IS LIVE (via localtunnel)!
Open this URL: https://clean-groups-relate.loca.lt
Share it with anyone — works on phone too!
